In [ ]:
import os
import numpy as np
import pandas as pd
import cv2

# import splitfolders
import h5py
from matplotlib import pyplot as plt
%matplotlib inline
from matplotlib import rcParams
import seaborn as sns
from PIL import Image
import imutils 

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.applications import (
    InceptionResNetV2,
    ResNet50,
    InceptionV3,
    DenseNet121,
)
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.imagenet_utils import preprocess_input
from tensorflow.keras.callbacks import TensorBoard



In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive/", force_remount=True)
    google_drive_prefix = "/content/drive/My Drive"
    data_prefix = "{}/mnist/".format(google_drive_prefix)
except ModuleNotFoundError: 
    data_prefix = "data/"

Mounted at /content/drive/


In [ ]:
!pip install wandb

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import wandb
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [ ]:
from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint

In [ ]:
# Start a run, tracking hyperparameters
wandb.init(
    # set the wandb project where this run will be logged
    project="Mermoire_2023_Version_01",

    # track hyperparameters and run metadata with wandb.config
    config={
        "dropout": 0.3,
        "dropout_2": 0.3,
        "activation": "softmax",
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 40,
        "batch_size": 32,
        "units_1": 128
    }
)

In [ ]:
config = wandb.config

In [ ]:
train_set = '/content/drive/My Drive/Datasets/Cropped_Image_Sets/train'
val_set = '/content/drive/My Drive/Datasets/Cropped_Image_Sets/val'
test_set = '/content/drive/My Drive/Datasets/Cropped_Image_Sets/test'
model_dir ="/content/drive/My Drive/Models/RadImageNet-ResNet50_notop.h5"
IMAGE_SIZE = 128

In [ ]:
def init_data(train_dir: str, valid_dir: str, test_dir: str) -> list:
    train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    valid_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rescale=1/255.0
    )
    
    train_data = train_datagen.flow_from_directory(
        directory=train_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    valid_data = valid_datagen.flow_from_directory(
        directory=valid_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    
    test_data = valid_datagen.flow_from_directory(
        directory=test_dir,
        class_mode='categorical',
        target_size=(128, 128),
        batch_size=config.batch_size,
        seed=42,
        shuffle=False,
    )
    
    return train_data, valid_data, test_data

In [ ]:
train_data, valid_data, test_data = init_data(train_dir=train_set, valid_dir=val_set, test_dir=test_set)

Found 2144 images belonging to 3 classes.
Found 458 images belonging to 3 classes.
Found 462 images belonging to 3 classes.


In [ ]:
model_name = "My_model"

TensorBoard = TensorBoard(log_dir="logs\\{}".format(model_name))

In [ ]:
def build_transfer_learning_model(base_model):
    # `base_model` stands for the pretrained model
    # We want to use the learned weights, and to do so we must freeze them
    for layer in base_model.layers:
        layer.trainable = False
        
    # Declare a sequential model that combines the base model with custom layers
    model = tf.keras.Sequential([
        base_model,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(rate=config.dropout),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(config.units_1, activation=config.activation),
        tf.keras.layers.Dropout(rate=config.dropout_2),
        tf.keras.layers.Dense(units=3, activation=config.activation)
    ])
    
    # Compile the model
    model.compile(
        loss=config.loss,
        optimizer=config.optimizer,
        metrics=[config.metric]
    )
    
    return model

In [ ]:
rad_model = build_transfer_learning_model(
    base_model=ResNet50(weights= model_dir, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),  include_top=False, pooling="avg")
)

In [ ]:
rad_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 2048)              23587712  
                                                                 
 flatten_1 (Flatten)         (None, 2048)              0         
                                                                 
 dropout_2 (Dropout)         (None, 2048)              0         
                                                                 
 batch_normalization (BatchN  (None, 2048)             8192      
 ormalization)                                                   
                                                                 
 dense_2 (Dense)             (None, 128)               262272    
                                                                 
 dropout_3 (Dropout)         (None, 128)               0         
                                                      

In [ ]:
# Train the model for 10 epochs
rad_hist = rad_model.fit(
    train_data,
    validation_data=valid_data,
    epochs=config.epoch,
    callbacks= [TensorBoard,
                WandbMetricsLogger(log_freq=5),
                WandbModelCheckpoint("models"),
                ]
)
wandb.finish()

Epoch 1/40
67/67 [==============================] - ETA: 0s - loss: 1.0843 - accuracy: 0.4487

wandb: Adding directory to artifact (./models)... Done. 0.6s


67/67 [==============================] - 662s 10s/step - loss: 1.0843 - accuracy: 0.4487 - val_loss: 1.0850 - val_accuracy: 0.4651
Epoch 2/40
67/67 [==============================] - ETA: 0s - loss: 1.0358 - accuracy: 0.5597

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 226s 3s/step - loss: 1.0358 - accuracy: 0.5597 - val_loss: 1.0642 - val_accuracy: 0.5240
Epoch 3/40
67/67 [==============================] - ETA: 0s - loss: 0.9988 - accuracy: 0.5942

...

67/67 [==============================] - 221s 3s/step - loss: 0.5740 - accuracy: 0.8092 - val_loss: 0.8066 - val_accuracy: 0.6834
Epoch 22/40
67/67 [==============================] - ETA: 0s - loss: 0.5612 - accuracy: 0.8176

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 221s 3s/step - loss: 0.5612 - accuracy: 0.8176 - val_loss: 0.8107 - val_accuracy: 0.6659
Epoch 23/40
67/67 [==============================] - ETA: 0s - loss: 0.5587 - accuracy: 0.8078

wandb: Adding directory to artifact (./models)... Done. 0.4s


67/67 [==============================] - 208s 3s/step - loss: 0.5587 - accuracy: 0.8078 - val_loss: 0.7866 - val_accuracy: 0.6965
Epoch 24/40
26/67 [==========>...................] - ETA: 1:30 - loss: 0.5583 - accuracy: 0.8185